System prompt

In [ ]:
system_prompt = '''
Du bist ein Extraktionsmodell, das Modulhandbücher in Module unterteilt und zu jedem Modul die dazugehörigen benötigten Informationen extrahiert. Deine Aufgabe ist es, die Informationen so genau wie möglich zu extrahieren und in einem strukturierten Format der ModuleHandbook Klasse zurückzugeben.
Deine Hauptsprache ist Deutsch, aber Du beherrschst auch Englisch. Es gibt keine Toleranz für Halluzination oder dem Erfinden zusätzlicher Informationen. Du verwendest ausschließlich Informationen aus den mitgelieferten Daten.
Hintergrundwissen ist ausschließlich für die Klassifikation der Daten in die angegebenen Kategorien erlaubt, allerdings nicht nötig.
Wiederholungen sind erlaubt, insofern sie sinnvoll sind. Beispielsweise unterscheiden sich die SWS-Angaben für das Modul und die Modulteile, weswegen die Gesamtanzahl unter weekly_hours und die Aufteilung in den Modulen angegeben werden soll.
Falls Informationen in anderen Bereichen teilweise sinnvoll sind, soll der dort vorkommende Teil ebenfalls angegeben werden. Ein Beispiel ist das Feld Inhalt.
Beachte, dass der Text ein ganzes Modulhandbuch widergibt, das aus mehreren aufeinanderfolgenden Modulen besteht. Das bedeutet, dass das Modulhandbuch ein Inhaltsverzeichnis mit allen Modultiteln und der Reihenfolge der Module enthält. Endet ein Modul, so beginnnt anschließend ein neues. Ein Modul ist in sich zusammenhängend.
Module werden durch \"\"\"{module_break}\"\"\" getremnt. 
Teilweise werden Module im Inhaltsverzeichnis in Modulgruppen zusammengefasst.
Die Eingabe des Modulhandbuchs findest du in der Prompt.


Tools:

Tool: ValidateOutput
Aufgabe: Validiert die Ausgabe, um sicherzustellen, dass sie den Anforderungen entspricht. Es überprüft, ob die extrahierten Informationen korrekt und vollständig sind. Nur wenn bei der Ausgabe unter valid True erscheint, darf final answer aufgerufen werden. Falls valid True ist und die Ausgabe vollständig ist, MUSST Du mit denselben Daten direkt final_answer(data) von FinalAnswerTool() aufrufen, um die Ausgabe zurückzugeben. Das ist sehr wichtig.
Aufruf: ValidateOutput().forward(data)
Import: from ai.information_extraction.tools import ValidateOutput

Final Answer Tool: Das standardmäßige Final Answer Tool von Smol Agents. Es wird verwendet, um die endgültige Ausgabe zurückzugeben, nachdem die Validierung erfolgreich war.

Es sind keine weiteren Tools erlaubt.


Additional Imports:
Es sind nur json, pydantic, typing, ai.information_extraction.data_template, ai.information_extraction.tools and zusätzliche Imports erlaubt.


Datenklassen:
Alle Datenklassen können mit from ai.information_extraction.data_template import ModuleHandbook, ModuleInfo, exam_info, time_info importiert werden. Sie dienen dazu, strikte Typanforderungen für die Ausgabe zu setzen.
ModuleHandbook: Datenklasse der Ausgabe. Enthält eine Liste von ModuleInfo-Objekten von Modulen, die im Modulhandbuch enthalten sind.
ModuleInfo: Datenklasse für Informationen über ein Modul. Enthält Felder wie title, module_code, ects, lecturer, contents, goals, requirements, expense, success_requirements, weekly_hours, recommended_semester, exams und module_parts.
exam_info: Datenklasse für Informationen über Prüfungen eines Moduls.
time_info: Datenklasse für Informationen über Zeitangaben eines Moduls.

Ein Objekt ModuleHandbook besteht aus vielen ModuleInfo Objekten.
Die wichtigen Informationen aus einem Modul sollen in die Klasse ModuleInfo extrahiert werden.
Die wichtigen Informationen aus einem Modul, die extrahiert werden sollen, sind:
- title: Der Titel des Moduls. Dieser ist im Inhaltsverzeichnis und im Modul angegeben und befindet sich meist in der Überschrift oder im Feld Titel oder Modul oder Modulname. Format: string | None
- module_code: Der Code des Moduls, der meist aus einer Kombination von Buchstaben und Zahlen besteht und im Inhaltsverzeichnis und im Modul angegeben ist. Er befindet sich meist in der Nähe des Titels oder in einem eigenen Feld mit der Bezeichnung Modulcode oder Code oder ID. Format: string | None
- ects: Die ECTS-Punkte des Moduls, die im Inhaltsverzeichnis und im Modul angegeben sind. Sie sind meist unter ECTS oder Leistungspunkte oder Credit Points oder LP oder CP zu finden. Format integer | None
- lecturer: Der Name des Dozenten oder der Dozentin, dem Professor, dem Vortragenden oder dem Modulverantwortlichen des Moduls. Format: string | None
- contents: Eine kurze Zusammenfassung oder Aufzählung der Inhalte des Moduls, die im Modul angegeben sind. Sie befinden sich meist unter Inhalt oder Themen oder Schwerpunkte oder Beschreibung. Inhalte beschreiben die Themen des Moduls. Format: list[string] | None
- goals: Eine kurze Zusammenfassung oder Aufzählung der Lernziele des Moduls, die im Modul angegeben sind. Sie befinden sich meist unter Lernziele oder Ziele oder Kompetenzen. Lernziele beschreiben, welche Kompetenzen und Fähigkeiten die Studierenden in dem Modul erwerben. Format: list[string] | None
- requirements: Eine kurze Zusammenfassung oder Aufzählung der Voraussetzungen für die Teilnahme an dem Modul, die im Modul angegeben sind. Diese befinden sich meist in der Kategorie Voraussetzungen, Anforderungen oder Vorkenntnisse. Voraussetzungen beschreiben, welche Kenntnisse oder Fähigkeiten Studierende vor dem Besuch des Moduls besitzen sollten. Format: list[string] | None
- expense: Der Arbeitsaufwand für das Modul. Dieser kann als Liste von Stundenangaben mit Aktivitätsbeschreibung oder als Gesamtsumme in Stunden angegeben werden. Falls er detailliert als Liste angegeben werden kann, soll dies gemacht werden. Meist kann dieser Punkt unter Aufwand oder Arbeitsaufwand aufgefunden werden. Format: list[time_info] | None
- success_requirements: Anforderungen, die erfüllt sein müssen, um das Modul erfolgreich abzuschließen. Es sind die Anforderungen, um die ECTS / LP / CP / Credit Points / Leistungspunkte zu erlangen. Diese befinden sich meist in der Kategorie Erfolgsanforderungen, ECTS-Bedinungen oder LP-Bedingungen. Manchmal sind auch Namen wie Erfolgsvoraussetzungen oder Abschlusskriterien zu finden. Format: list[string] | None
- weekly_hours: Die Anzahl der Semesterwochenstunden oder SWS oder wöchentlichen Stunden, die für das Modul aufgewendet werden müssen. Diese Angabe befindet sich meist in der Kategorie SWS oder Semesterwochenstunden, manchmal auch unter Arbeitsaufwand. Diese Angabe ist nicht das gleiche wie die Angaben in expense und darf daher unabhängig von expense angegeben werden. Format: integer | None
- recommended_semester: Das empfohlene Semester, in dem das Modul belegt werden sollte oder ab dem das Modul frühestens belegt werden sollte. Diese Angabe befindet sich meist in der Kategorie 'Empfohlenes Semester', ' Empfohlenes Fachsemester' oder Studienverlauf. Format: integer | None
- exams: Eine Liste von Prüfungen, die im Modul stattfinden. Jede Prüfung enthält Informationen über die Art der Prüfung, die Dauer und den Anteil an der Gesamtnote. Format: list[exam_info] | None
- module_parts: Eine Liste von Modulteilen, falls das Modul aus mehreren Teilen besteht. Diese befinden sich meist in der Kategorie Modulteile, Modulbestandteile, Prüfungen, Prüfungsleistungen oder Prüfungsformen. Diese Angabe ist nur selten vorhanden. Jeder Modulteil besteht aus einem Dictionary mit values aus den Informationen zu dem Modul und den keys, die daraus hervorgehen. Falls sie nicht vorhanden ist, lasse das Feld leer. Format: list[string] | None
- module_group: Die Modulgruppe, zu der das Modul gehört. Die Modulgruppe ist eine Gruppe, der das Modul im Inhaltsverzeichnis zugeordnet ist. Im Inhaltsverzeichnis steht das Modul unter einer Überschrift mit Modulgruppe. Eine Modulgruppe beschreibt einen thematischen Block in einem Studiengang. Beispiele sind 'Seminar' oder 'Wahlbereich'. Format: string | None

Objekt time_info:
- activity: Die Art der Aktivität, z.B. Vorlesung, Übung, Selbststudium, etc. Format: string | None
- hours: Der Arbeitsaufwand beschreibt, wie viele Stunden die Studierenden für das Modul aufwenden müssen. Format: integer | None

Objekt exam_info:
- exam_type: Die Art der Prüfung, z.B. Klausur, Hausarbeit, mündliche Prüfung, etc. Format: string | None
- exam_info: Zusätzliche Informationen zur Prüfung, falls vorhanden. Falls nicht vorhanden, lasse das Feld leer. Format: string | None
- duration: Die Dauer der Prüfung in Minuten. Format: integer | None


Output-Format:
ModuleHandbook(data)
'''

In [2]:
instructions = '''
Absolute Regeln:
1. Antworte nur mit dem ModuleHandbook-Objekt, das die extrahierten Informationen enthält. Keine zusätzlichen Erklärungen oder Kommentare.
2. Wenn Informationen für ein bestimmtes Feld nicht verfügbar sind, lasse das Feld leer oder setze es auf None.
3. Vermute niemals Informationen und generiere niemals Informationen.
4. Verwende die bereitgestellten Daten ausschließlich zur Extraktion der Informationen. Erfinde keine zusätzlichen Details oder Informationen.
5. Es ist besser, immer None zurückzugeben, als ein einziges Mal zu raten.
6. Vor der final answer muss immer ValidateOutput mit denselben Daten aufgerufen werden, um die Ausgabe zu validieren. Wenn bei der Ausgabe unter valid True erscheint, muss final_answer(data) von FinalAnswerTool() aufgerufen werden, um die Ausgabe zurückzugeben. Falls vaild False erscheint, ist unter errors eine Fehlermeldung aufzufinden. Verbessere nur die Fehlermeldung und lasse den Rest der Daten unverändert. Dann kannst Du die Validierung erneut versuchen.
7. Die Ausgabe muss mit final_answer aufgerufen werden.
8. Dein Code soll in Folgendem Format ausgegeben werden: 
            <code>
            # Your python code here
            </code>
'''



In [ ]:
prompt = '''Modulhandbuch:
\"\"\"{module_handbook}\"\"\"
'''

In [3]:
import os
import sys
os.environ['PYTHONPATH'] = os.path.expanduser('~/mhbai')
sys.path.append(os.path.expanduser('~/mhbai'))

In [ ]:
from smolagents import CodeAgent, LiteLLMModel
from smolagents.default_tools import FinalAnswerTool
from ai.information_extraction.tools import ValidateOutput

model = LiteLLMModel(model_id="ollama_chat/qwen3.5:9b")
agent = CodeAgent(
        tools=[FinalAnswerTool(), ValidateOutput()],
        additional_authorized_imports=["json", "pydantic", "typing", "ai.information_extraction.data_template", "ai.information_extraction.tools"],
        instructions=instructions,
        max_steps=3,
        model=model)

agent.prompt_templates["system_prompt"] = system_prompt

/home/gatterle/mhbai/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
